# Predicting war events

## I. Importing essential libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
import json

## II. Importing

### ISW

In [2]:
json_path_isw = Path("../data/isw_reports.json")

with open(json_path_isw, "r", encoding="utf-8") as f:
    data = json.load(f)

df_isw_raw = pd.DataFrame(data)

print(type(df_isw_raw))

<class 'pandas.core.frame.DataFrame'>


## III. EDA

### ISW

In [3]:
df_isw_raw.shape

(1439, 4)

In [4]:
df_isw_raw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [5]:
df_isw_raw.sample(5)

,date,title,url,text
1398,2026-01-20,"Russian Offensive Campaign Assessment, January...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
871,2024-08-02,"Russian Offensive Campaign Assessment, August ...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
415,2023-04-28,"Russian Offensive Campaign Assessment, April 2...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
43,2022-04-14,"Russian Offensive Campaign Assessment, April 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
1281,2025-09-21,"Russian Offensive Campaign Assessment, Septemb...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [6]:
df_isw_raw.describe()

,date,title,url,text
count,1439,1439,1439,1439
unique,1439,1439,1439,1439
top,2026-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
freq,1,1,1,1


In [7]:
df_isw_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1439 entries, 0 to 1438
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    1439 non-null   object
 1   title   1439 non-null   object
 2   url     1439 non-null   object
 3   text    1439 non-null   object
dtypes: object(4)
memory usage: 45.1+ KB


The dataset contains 1439 rows and 4 columns. 
All values are unique.  
All columns currently have the `object` data type. 
No missing values were found

In [8]:
df_isw = df_isw_raw.copy()

In [9]:
df_isw["date"] = pd.to_datetime(df_isw["date"], errors="coerce")
print("Invalid dates:", df_isw["date"].isna().sum())
print(df_isw["date"].dtype)

Invalid dates: 0
datetime64[ns]


In [10]:
print("Duplicate dates:", df_isw["date"].duplicated().sum())

Duplicate dates: 0


In [11]:
all_dates = pd.date_range("2022-02-24", "2026-03-01")
existing_dates = pd.DatetimeIndex(df_isw["date"])
missing_dates = all_dates.difference(existing_dates)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 28
DatetimeIndex(['2022-02-24', '2022-03-16', '2022-03-18', '2022-03-29',
               '2022-04-01', '2022-04-11', '2022-05-14', '2022-06-23',
               '2022-10-28', '2022-11-24', '2022-12-25', '2023-01-01',
               '2023-03-19', '2023-07-08', '2023-07-27', '2023-11-23',
               '2023-12-25', '2024-01-01', '2024-01-05', '2024-10-08',
               '2024-11-28', '2024-12-25', '2025-01-01', '2025-09-07',
               '2025-10-31', '2025-11-27', '2025-12-25', '2026-01-01'],
              dtype='datetime64[ns]', freq=None)


In [12]:
outside_range = df_isw[(df_isw["date"] < "2022-02-24") | (df_isw["date"] > "2026-03-01")]
print(outside_range)

Empty DataFrame
Columns: [date, title, url, text]
Index: []


In [13]:
def clean_isw_text(text):
    text = str(text)
    text = text.replace("Previous\nNext", "")
    text = text.replace("\n", " ")
    return text

df_isw["text_clean"] = df_isw["text"].apply(clean_isw_text)

In [14]:
print("=== RAW TEXT:")
print(df_isw.loc[0, "text"])

print("\n" + "="*100 + "\n")

print("=== CLEAN TEXT:")
print(df_isw.loc[0, "text_clean"])

=== RAW TEXT:
Previous
Next
Russia-Ukraine Warning Update: Russian Offensive Campaign Assessment, February 25, 2022
Mason Clark, George Barros, and Kateryna Stepanenko
February 25, 3:00 pm EST
Russian forces entered major Ukrainian cities—including Kyiv and Kherson—for the first time on February 25.
Russian forces’ main axes of advance focused on Kyiv (successfully isolating the city on both banks of the Dnipro River). Russian military operations along Ukraine’s northern border have been less well-planned, organized, and conducted than those emanating from Crimea. They have also been less successful so far. The divergence in performance likely arises in part from differences in the composition and organization of the Russian ground forces elements in the Western Military District and Belarus (to Ukraine’s north) and Southern Military District and Black Sea Fleet (to its south and east), as ISW has previously observed.
[1]
Determined and well-organized Ukrainian resistance around Kyiv a

In [15]:
df_isw["text"] = df_isw["text_clean"]
df_isw = df_isw.drop(columns=["text_clean"])

In [16]:
df_isw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, Februa..."
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, March ..."


## IV. Prepare data